In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from numpy import loadtxt
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
import pickle

In [2]:
data = pd.read_csv('Train_Loan.csv') 

In [3]:
data.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [4]:
data.isnull().sum()  

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [5]:
 data = data.fillna(data.mean().iloc[0]) 

C:\Users\Admin\AppData\Local\Temp\ipykernel_3768\2001788651.py:1: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  data = data.fillna(data.mean().iloc[0])


In [6]:
for col in data.columns:
    if data[col].dtypes == 'object':
        num_of_unique_cat = len (data[col].unique())
        print("Features '{col_name}' has '{unique_cat}' unique categories". format(col_name=col, unique_cat=num_of_unique_cat))

Features 'Loan_ID' has '614' unique categories
Features 'Gender' has '3' unique categories
Features 'Married' has '3' unique categories
Features 'Dependents' has '5' unique categories
Features 'Education' has '2' unique categories
Features 'Self_Employed' has '3' unique categories
Features 'Property_Area' has '3' unique categories
Features 'Loan_Status' has '2' unique categories


In [7]:
data = data.drop(['Loan_ID','Dependents', 'Gender','Married','Education','Self_Employed','Property_Area'], axis = 1)


In [8]:
data['Loan_Status'].replace({'Y':1,'N':0},inplace=True)

In [9]:
y = data['Loan_Status'] 
X = data.drop('Loan_Status', axis = 1) 

In [10]:
from sklearn.preprocessing import StandardScaler
sc_X = StandardScaler()
X_scaled = pd.DataFrame(sc_X.fit_transform(X), columns=X.columns)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

In [13]:
from sklearn.linear_model import LogisticRegression


model_lr = LogisticRegression(solver = 'liblinear', random_state = 42)
model_lr.fit(X_train, y_train)
ypred = model_lr.predict(X_test) 
evaluation = f1_score(y_test, ypred)
evaluation

0.838095238095238

In [16]:
from sklearn import svm

model_sv = svm.SVC(kernel='linear')

model_sv.fit(X_train, y_train)
ypred = model_sv.predict(X_test) 
evaluation = f1_score(y_test, ypred)
evaluation

0.8309178743961353

In [17]:
from lightgbm import LGBMClassifier

model_lgbm = LGBMClassifier()
model_lgbm.fit(X_train, y_train) 
ypred = model_lgbm.predict(X_test) 
evaluation = f1_score(y_test, ypred)
evaluation


0.8695652173913044

In [19]:
from sklearn.ensemble import RandomForestClassifier

model_rfc = RandomForestClassifier()
model_rfc.fit(X_train, y_train) 
ypred = model_rfc.predict(X_test) 
evaluation = f1_score(y_test, ypred)
evaluation

0.858695652173913

In [ ]:
classifiers = []
classifiers.append(model_sv)
classifiers.append(model_lgbm)
classifiers.append(model_rfc)
classifiers.append(model_lr)

In [ ]:
accuracy_list = []

for classifier in classifiers:
    y_pred = classifier.predict(X_test)
    accuracy_list.append(f1_score(y_test, y_pred))
    

accuracy_dict = {}

for i in range(4):
    key=['SVM', 'Light GBM', 'Random Forest','Logistic Regression'][i]
    accuracy_dict[key] = accuracy_list[i]
    
    
accuracy_dict_sorted = dict(sorted(accuracy_dict.items(), key = lambda item: item[1]))

In [ ]:
def px_bar(x,y,text,title,color,color_discrete_sequence):
    return px.bar(x = x, y = y, text = text, title = title, color = color, color_discrete_sequence=color_discrete_sequence)

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

fig = px_bar(list(accuracy_dict_sorted.keys()), list(accuracy_dict_sorted.values()), np.round(list(accuracy_dict_sorted.values()),3), 'Accuracy score of each classifiers', list(accuracy_dict_sorted.keys()), px.colors.sequential.matter)
for idx in [2,3]:
    fig.data[idx].marker.line.width = 3
    fig.data[idx].marker.line.color = "black"
fig.show()

In [20]:
pickle.dump(model_lgbm,open('model2.pkl','wb'))